In [1]:
import numpy as np
import pandas as pd
import seaborn as sns


from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.mixture import GaussianMixture
from sklearn.metrics import mean_absolute_error
from nltk.cluster.kmeans import KMeansClusterer
from nltk.cluster.util import euclidean_distance
from sklearn.impute import KNNImputer
from sklearn.decomposition import PCA
from ucimlrepo import fetch_ucirepo
from imblearn.under_sampling import RandomUnderSampler

# fetch dataset
support2 = fetch_ucirepo(id=880)

import umap.umap_ as umap
import matplotlib.pyplot as plt

from models_II import *
from my_encodings import *


In [2]:
def get_dataset():
    # data (as pandas dataframes)
    features = support2.data.features
    targets = support2.data.targets

    # join together as 1 df
    df = features.join(targets)
    df = df.drop(columns=['death', 'hospdead', 'adlp', 'adls', 'scoma', 'totmcst', 'totcst', 'sps', 'aps', 'hday', \
                        'adlsc', 'prg2m', 'prg6m', 'charges', 'dzgroup', 'dnr', 'dnrday', 'urine',
                        'surv2m', 'surv6m']) # added surv2m, surv6m to drop; prevent data leakage

    # drop rows where target = NaN
    print(f'Before dropping NaNs in target: {df.shape[0]}')
    df = df.dropna(subset=['sfdm2'])
    print(f'After dropping NaNs in target: {df.shape[0]}')

    return

In [3]:
def get_encodings(df: pd.DataFrame):
    ''' combines target classes to improve class imbalance, encodes categorical features via dummy or ordinal encoding'''
    # isolate categorical features
    df_to_encode = df.copy()
    numerical_feats = df.select_dtypes(include=['number']).columns.tolist() # determine numerical columns
    df_to_encode = df_to_encode.drop(numerical_feats, axis=1) # drop numerical columns

    # ------------ DUMMY ENCODING -----------
    dum_cols = ['sex', 'dzclass', 'race', 'ca']
    df_to_encode = pd.get_dummies(df_to_encode, columns=dum_cols, dtype=int)

    # ------------ ORDINAL ENCODING -----------
    income_order = {
        'under $11k': 0,
        '$11-$25k': 1,
        '$25-$50k': 2,
        '>$50k': 3
    }

    df_to_encode['income'] = df_to_encode['income'].map(income_order)

    target_order = {
        'no(M2 and SIP pres)': 0, # 0 = healthy
        'adl>=4 (>=5 if sur)': 1, # 1 = moderate disability
        'SIP>=30': 1,
        'Coma or Intub': 1,
        '<2 mo. follow-up': 2     # 2 = death
    }

    df_to_encode['sfdm2'] = df_to_encode['sfdm2'].map(target_order)

    # recombine dataframe
    df = pd.concat([df_to_encode, df[numerical_feats]], axis=1)

    # examine class imbalance
    counts = df['sfdm2'].value_counts()
    plt.bar(counts.index, counts.values)
    plt.xlabel("Classes")
    plt.ylabel("Counts")
    plt.title("Class Distribution after Rebalancing classes")
    plt.show()

    return df

In [ ]:
def split_scale_impute(df):
    # -------------------------------------- Split ----------------------------------
    X = df.drop(columns=['sfdm2'])
    y = df[['sfdm2']]

    # split into training vs test data
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True, random_state=42)

    # -------------------------------------- Scale all Features ----------------------------------
    scaler = StandardScaler()
    XS_train = scaler.fit_transform(X_train)
    XS_test = scaler.transform(X_test)

    # -------------------------------------- Impute ----------------------------------
    imputer = KNNImputer(n_neighbors=10) # initialize imputer 

    # training
    XS_train = imputer.fit_transform(XS_train) # fit and transform training set 

    # testing
    XS_test = imputer.transform(XS_test)

    # -------------------------------------- NaN Check ----------------------------------
    train_df = pd.DataFrame(XS_train)
    test_df = pd.DataFrame(XS_test)

    print(f"Contains missing values xtrain: {train_df.isna().any().any()}")
    print(f"Contains missing values xtest: {test_df.isna().any().any()}")
    print(f"Contains missing values ytrain: {y_train.isna().any().any()}")
    print(f"Contains missing values ytest: {y_test.isna().any().any()}")
    return XS_test, XS_train